<style>
:root {
  --jp-layout-color0: #1e1e2e;
  --jp-layout-color1: #181825;
  --jp-layout-color2: #313244;
  --jp-content-font-color0: #cdd6f4;
  --jp-content-font-color1: #cdd6f4;
  --jp-content-font-color2: #a6adc8;
}
body, .jp-Notebook { background: #1e1e2e !important; color: #cdd6f4 !important; }
.jp-Cell { background: #1e1e2e !important; }
.jp-CodeCell .jp-Cell-inputWrapper, .jp-CodeMirrorEditor { background: #181825 !important; }
.cm-s-jupyter { background: #181825 !important; color: #cdd6f4 !important; }
.jp-MarkdownOutput, .jp-RenderedMarkdown { color: #cdd6f4 !important; }
</style>

# Learning to Play Blackjack with Reinforcement Learning

## Install and Import Dependencies

In [1]:
# Install gymnasium if needed
# !pip install gymnasium matplotlib numpy seaborn

import random
import math
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from tqdm import tqdm

print(f"Gymnasium version: {gym.__version__}")

Gymnasium version: 1.2.3


## Set Up and Explore the Blackjack Environment

The Blackjack-v1 environment simulates the card game with the following:
- **State**: A tuple of (player_sum, dealer_showing_card, usable_ace)
  - `player_sum` (int): Current sum of player's hand (4–21)
  - `dealer_showing_card` (int): Dealer's face-up card (1–10, where 1 = Ace)
  - `usable_ace` (bool): Whether the player has a usable ace
- **Actions**: 0 = Stand (stop), 1 = Hit (draw another card)
- **Rewards**: +1 (win), 0 (draw), -1 (lose)

In [2]:
# Create the environment
env = gym.make('Blackjack-v1')  # sab=True uses Sutton & Barto rules

print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Actions: 0=Stand, 1=Hit")

Observation space: Tuple(Discrete(32), Discrete(11), Discrete(2))
Action space: Discrete(2)
Actions: 0=Stand, 1=Hit


In [3]:
# Play a few sample episodes manually to understand the environment
print("=== Sample Episodes ===")

for episode in range(5):
    state, info = env.reset()
    print(f"\nEpisode {episode + 1}:")
    print(f"  Initial state: player_sum={state[0]}, dealer_card={state[1]}, usable_ace={state[2]}")
    
    done = False
    step = 0
    while not done:
        # Random action for demonstration
        action = env.action_space.sample()
        action_name = 'Hit' if action == 1 else 'Stand'
        
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        step += 1
        
        print(f"  Step {step}: Action={action_name}, "
              f"New state={next_state}, Reward={reward}, Done={done}")
        
        state = next_state
    
    result = 'Win' if reward > 0 else ('Draw' if reward == 0 else 'Lose')
    print(f"  Result: {result}")

=== Sample Episodes ===

Episode 1:
  Initial state: player_sum=9, dealer_card=10, usable_ace=0
  Step 1: Action=Stand, New state=(9, 10, 0), Reward=-1.0, Done=True
  Result: Lose

Episode 2:
  Initial state: player_sum=15, dealer_card=1, usable_ace=0
  Step 1: Action=Stand, New state=(15, 1, 0), Reward=-1.0, Done=True
  Result: Lose

Episode 3:
  Initial state: player_sum=16, dealer_card=6, usable_ace=0
  Step 1: Action=Stand, New state=(16, 6, 0), Reward=1.0, Done=True
  Result: Win

Episode 4:
  Initial state: player_sum=15, dealer_card=1, usable_ace=0
  Step 1: Action=Hit, New state=(25, 1, 0), Reward=-1.0, Done=True
  Result: Lose

Episode 5:
  Initial state: player_sum=20, dealer_card=10, usable_ace=0
  Step 1: Action=Hit, New state=(30, 10, 0), Reward=-1.0, Done=True
  Result: Lose


## Random Policy Baseline

We've provided two simple baselines:

1. Random policy agent: This agent will choose to hit or stand randomly.

2. Stand on 17+: This agent will hit until it has reached 17+ in which it will stand.

**Note:** Your job is to implement better RL algorithms to beat these baselines.

In [4]:
def evaluate_policy(env, policy_fn, n_episodes=100000):
    """
    Evaluate a policy over many episodes.
    
    Args:
        env: Gymnasium environment
        policy_fn: Function that takes a state and returns an action
        n_episodes: Number of episodes to simulate
    
    Returns:
        win_rate, draw_rate, lose_rate
    """
    wins, draws, losses = 0, 0, 0
    
    for _ in range(n_episodes):
        state, _ = env.reset()
        done = False
        
        while not done:
            action = policy_fn(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
        
        if reward > 0:
            wins += 1
        elif reward == 0:
            draws += 1
        else:
            losses += 1
    
    return wins / n_episodes, draws / n_episodes, losses / n_episodes


# Random policy
random_policy = lambda state: env.action_space.sample()
win_rate, draw_rate, lose_rate = evaluate_policy(env, random_policy)
print(f"Random Policy: Win={win_rate:.3%}, Draw={draw_rate:.3%}, Lose={lose_rate:.3%}")

# Simple threshold policy (stand on 17+)
threshold_policy = lambda state: 1 if state[0] < 17 else 0
win_rate, draw_rate, lose_rate = evaluate_policy(env, threshold_policy)
print(f"Threshold (17) Policy: Win={win_rate:.3%}, Draw={draw_rate:.3%}, Lose={lose_rate:.3%}")

Random Policy: Win=28.275%, Draw=4.153%, Lose=67.572%
Threshold (17) Policy: Win=40.878%, Draw=10.330%, Lose=48.792%


In [5]:
# TODO: Your code goes here

# data will have 3 features
# [playersum, dealercard, usableace]
# trying to predict action (0 or 1)

class blackjack_learner_perceptron():
    """
    Old Model from previous submission
    """
    weights = [0, 0, 0]
    bias = 0
    zscores = []
    data = []
    correct_action = []
    p = []
    loss = []
    gradients = []
    learning_rate = 0.001

    def __init__(self, weights = [0, 0, 0], bias = 0, data = []):
        self.weights = weights
        self.bias = bias
        self.data = data

    def play(self, inputs):
        z = 0
        for i in range(len(inputs)):
            z += self.weights[i] * inputs[i]
        z += self.bias

        if z > 0:
            return 1
        return 0

    def update(self, env, inputs):
        # Takes in a game
        # Randomly guesses whether to hit or stand
        # If it hits and loses, or stands and loses, then the "correct" option is labelled as the opposite action
        # If it stands and wins, the correct option is labelled as standing
        # If it hits doesn't bust, the new state is passed into the update function again
        # Then if the next update function wins, hitting is labelled as correct
        # If it doesn't win, this state is thrown away and not put into the dataset (We don't know if the initial hit was correct or not!)

        # If it stands and draws, then standing is labelled as the correct action (Has to have at least 17 in order to stand and draw)
        # If it hits and then draws, the hit is labelled as a correct action

        action = random.choice([0, 1])
        new_inputs, reward, terminated, truncated, _ = env.step(action)
        new_inputs = list(new_inputs)
        done = terminated or truncated

        if(done):
            if(reward > 0):
                # Can only be here if we won by standing
                self.correct_action.append(action)
                self.data.append(inputs)
                return(reward)
            elif(reward == 0):
                # Can only be here if we tied by standing
                self.correct_action.append(action)
                self.data.append(inputs)
                return(reward)
            else:
                # Can be here if we stood and lost or hit and busted
                self.correct_action.append(self.invert_choice(action))
                self.data.append(inputs)
                return(reward)
        else:
            reward = self.update(env, new_inputs)
            if(reward == 0):
                # Can only be here if we hit, and then stood to draw
                self.correct_action.append(action)
                self.data.append(inputs)
                return reward
            elif(reward > 0):
                # Can only be here if the we hit, and then stood to win
                self.correct_action.append(action)
                self.data.append(inputs)
                return reward
            else:
                # Can be here if we hit, then hit again to bust or stood later and lost
                # We don't know if the action in this iteration was correct or not, so we will just throw this away for now
                return reward
            
                
        return
    def invert_choice(self, num):
        # returns inverse of an action
        if num == 1:
            return 0
        return 1
    def generate_data(self, epochs):
        # Use the update function to generate a dataset with Playersum, dealercard, usableace, and correct action
        # We should have a different number of points than our actual epochs because some points get thrown out, while others will
        # result in multiple actions meaning multiple points
        i = 0
        self.data = []
        # Not making sab = true because it will complicate things for our perceptron function
        curr_env = gym.make('Blackjack-v1')
        while i < epochs:
            state, _ = curr_env.reset()
            self.update(curr_env, list(state))
            i += 1
    def generate_zscores(self):
        self.zscores = []
        for observation in self.data:
            total = self.bias + 0
            for i in range(len(observation)):
                total += observation[i] * self.weights[i]
            self.zscores.append(total)
            
    def stable_sigmoid(self, z):
        """
        This implementation avoids overflow issues by handling large positive and negative values of z separately.
    
        z: a numeric value.

        if z >= 0: use the 'standard' formula: 1/(1 + exp(-z))
        if z < 0: use the alternative formula to avoid overflow: exp(z) / (1 + exp(z))
        """
        # Imported from lecture notebook
        if z >= 0:
            return 1 / (1 + math.exp(-z))
        else:
            ez = math.exp(z)
            return ez / (1 + ez)
        
    def generate_probability(self):
        self.p = []
        for z in self.zscores:
            self.p.append(self.stable_sigmoid(z))
    def generate_loss(self):
        self.loss = []
        for i in range(len(self.p)):
            if self.correct_action[i] == 1:
                self.loss.append(-1 * np.log(self.p[i]))
            else:
                self.loss.append(-1 * np.log(1 - self.p[i]))
    def generate_gradients(self):
        self.gradients = [0, 0, 0, 0]
        for i in range(len(self.correct_action)):
            for j in range(len(self.data[i])):
                self.gradients[j] += ((self.p[i] - self.correct_action[i]) * self.data[i][j])
            self.gradients[3] += (self.p[i] - self.correct_action[i])
            # Setting gradient for bias
        for i in range(len(self.gradients)):
            self.gradients[i] = self.gradients[i] / len(self.data)
    def set_learning_rate(self, newrate):
        self.learning_rate = newrate
    def nudge_weights(self):
        for i in range(len(self.weights)):
            self.weights[i] = self.weights[i] - self.gradients[i] * self.learning_rate
        self.bias = self.bias - self.gradients[3] * self.learning_rate
    def forward(self):
        self.generate_zscores()
        self.generate_probability()
        self.generate_gradients()
        self.nudge_weights()
    def learn(self, epochs):
        for i in range(epochs):
            self.forward()
            if i % 1000 == 0:
                self.generate_loss()
                print(f"Epoch: {i}")
                print(f"Average Loss: {sum(self.loss) / len(self.loss)}")

In [37]:
myLearner = blackjack_learner_perceptron()
myLearner.generate_data(10000)
# Generating a dataset with more than 10k blackjack games
myLearner.learn(5000)

Epoch: 0
Average Loss: 0.6931471805600621
Epoch: 1000
Average Loss: 0.6457976488075204
Epoch: 2000
Average Loss: 0.6439300921380333
Epoch: 3000
Average Loss: 0.6423223829823265
Epoch: 4000
Average Loss: 0.6407549784604994


# New model, utilizing reinforcement Q-learning

In [8]:
def get_q_value(Q, state, action) -> float:
    """Return Q(state, action), defaulting to 0.0 if missing."""
    # TODO
    if (state, action) in Q.keys():
        return Q[(state,action)]
    return 0

In [29]:
def best_future_reward(Q, state) -> float:
    """
    Return max_a Q(state, a).
    If terminal (or no legal actions), return 0.0.
    """
    # TODO
    largest = 0.0 
    found_any = False
    
    for tup in Q.keys():
        if tup[0] == state:
            if not found_any or Q[tup] > largest:
                largest = Q[tup]
                found_any = True
                
    return largest

In [30]:
def sarsa_update(
    Q,
    state,
    action,
    reward: float,
    next_state,
    next_action,
    alpha: float,
    gamma: float
) -> None:
    """
    SARSA update:
      Q(s,a) <- Q(s,a) + alpha * (reward + gamma * Q(s',a') - Q(s,a))
    If next_action is None (terminal), use bootstrap term 0.
    """
    # TODO
    if next_action == None:
        nextQ = 0
    else:
        nextQ = get_q_value(Q, next_state, next_action)
    x_t = reward + gamma * nextQ
    curr = get_q_value(Q, state, action)
    new_avg = curr + alpha * (x_t - curr)
    Q[(state, action)] = new_avg 

In [31]:
def choose_action(
    Q,
    state,
    epsilon: float
):
    """
    Epsilon-greedy action selection:
    - with probability epsilon: random legal action
    - otherwise: action with highest Q-value (break ties randomly)
    """
    # TODO
    greed = random.random()
    actions = [0, 1]
    if greed < epsilon:
        choice = random.choice(actions)
    else:
        largest = best_future_reward(Q, state)
        possible = []
        for action in actions:
            if get_q_value(Q, state, action) == largest:
                possible.append(action)
        choice = random.choice(possible)

    return choice

In [32]:
def train_episode(
    Q,
    alpha: float,
    gamma: float,
    epsilon: float,
) -> int:
    """
    Run one training episode.
    Return 1 if the agent wins, else 0.
    """
    check = False
    state = env.reset()[0]
    action = choose_action(Q, state, epsilon)
    while not check:
        newState, newReward, newTerminated, newTruncated, _ = env.step(action)
        
        if newTerminated | newTruncated:
            sarsa_update(Q, state, action, newReward, newState, next_action = None, alpha = alpha, gamma = gamma)
            if newReward > 0:
                return 1
            return 0
        nextAction = choose_action(Q, newState, epsilon)
        sarsa_update(Q, state, action, reward, newState, nextAction, alpha, gamma)
        state = newState
        action = nextAction

In [33]:
Q_sarsa = {}
outcome = train_episode(Q_sarsa, alpha=0.5, gamma=0.9, epsilon=0.2)
print('SARSA episode outcome:', outcome)
print('SARSA Q-table size:', len(Q_sarsa))

SARSA episode outcome: 0
SARSA Q-table size: 3


In [35]:
def train_sarsa_learning(
    episodes: int,
    alpha: float,
    gamma: float,
    epsilon: float,
    opponent_mode: str = 'random'
):
    Q = {}
    win_hist = []
    for i in range(episodes):
        won = train_episode(Q, alpha, gamma, epsilon)
        win_hist.append(won)
    return((Q, win_hist))
Q_small, win_hist_small = train_sarsa_learning(episodes=20000, alpha=0.02, gamma=0.9, epsilon=0.1)
print('Q-table size:', len(Q_small))
print('Last 10 outcomes:', win_hist_small[-10:])
sum(win_hist_small) / len(win_hist_small)

Q-table size: 517
Last 10 outcomes: [0, 0, 0, 0, 0, 0, 1, 0, 0, 1]


0.36895

In [44]:
def play_match(state):
    """
    Play one game with trained agent policy (greedy, no exploration).
    Return 1 if agent wins, else 0.
    """
    # TODO
    actions = [0, 1]
    largest = best_future_reward(Q_small, state)
    possible = []
    for action in actions:
        if get_q_value(Q_small, state, action) == largest:
            possible.append(action)
    choice = random.choice(possible)

    return choice

In [46]:
win_rate, draw_rate, lose_rate = evaluate_policy(env, play_match)
print(f"Threshold (17) Policy: Win={win_rate:.3%}, Draw={draw_rate:.3%}, Lose={lose_rate:.3%}")
# New Reinforcement Model

Threshold (17) Policy: Win=38.519%, Draw=4.952%, Lose=56.529%


In [47]:
win_rate, draw_rate, lose_rate = evaluate_policy(env, myLearner.play)
print(f"Threshold (17) Policy: Win={win_rate:.3%}, Draw={draw_rate:.3%}, Lose={lose_rate:.3%}")
# Old perceptron model

Threshold (17) Policy: Win=41.345%, Draw=7.111%, Lose=51.544%


:(